In [0]:
dbutils.widgets.text("tabela", "customers")
dbutils.widgets.text("chave_merge", "customer_id")

tabela = dbutils.widgets.get("tabela")
chave_merge = dbutils.widgets.get("chave_merge")

In [0]:
import delta

df_full = spark.read.format("parquet").load(f"/Volumes/projeto_olist/olist/fullload/{tabela}_fullload/")
df_full.display()

(df_full.coalesce(1)
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"projeto_olist.bronze.{tabela}_fullload"))

In [0]:
df_cdc = spark.read.format("parquet").load(f"/Volumes/projeto_olist/olist/cdc/{tabela}/")
df_cdc.createOrReplaceTempView(tabela)

In [0]:
query = f'''
select * from {tabela}
qualify row_number() over (partition by {chave_merge} order by data_particao desc) = 1
'''

df_cdc_unique = spark.sql(query)
df_cdc_unique.display()

In [0]:
bronze = delta.DeltaTable.forName(spark, f"projeto_olist.bronze.{tabela}_fullload")

(bronze.alias("b")
      .merge(df_cdc_unique.alias("d"), f"b.{chave_merge} = d.{chave_merge}")
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute())